# Module 3: Sorting Algorithms — Verified

**Prerequisites:** Modules 1–2

Sorting is a cornerstone of CS. In this module, we don't just *implement* sorting algorithms — we **prove** they are correct. In ACL2, a correct sorting algorithm must satisfy two properties:

1. **Ordered output:** The result is sorted (predicate `orderedp`)
2. **Permutation:** The output contains exactly the same elements as the input

We study four algorithms from the ACL2 community books (`books/sorting/`):
- Insertion sort (`isort.lisp`)
- Merge sort (`msort.lisp`)
- Bubble sort (`bsort.lisp`)
- Quicksort (`qsort.lisp`)

**Learning Objectives:**
1. Define and verify insertion sort
2. Understand the `perm` predicate for permutation equivalence
3. Implement and verify merge sort with a termination measure
4. Compare bubble sort and quicksort approaches
5. Prove that all sorting algorithms are equivalent

## 1. What Makes a Sort Correct?

A function `sort` is a correct sorting algorithm if for every input list `x`:

$$\text{orderedp}(\text{sort}(x)) \quad \wedge \quad \text{perm}(\text{sort}(x), x)$$

Let's define the building blocks.

In [1]:
; An ordered list: each element ≤ the next
; Uses ACL2's built-in lexorder for comparison
(defun orderedp (x)
  (cond ((endp x) t)
        ((endp (cdr x)) t)
        (t (and (lexorder (car x) (cadr x))
                (orderedp (cdr x))))))


The admission of ORDEREDP is trivial, using the relation O< (which
is known to be well-founded on the domain recognized by O-P) and the
measure (ACL2-COUNT X).  We observe that the type of ORDEREDP is described
by the theorem (OR (EQUAL (ORDEREDP X) T) (EQUAL (ORDEREDP X) NIL)).

Summary
Form:  ( DEFUN ORDEREDP ...)
Rules: NIL


orderedp

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)


In [2]:
; Test orderedp
(orderedp '(1 2 3 4 5))

t

In [3]:
(orderedp '(1 3 2 4 5))

nil

In [4]:
; Element count — how many times does e appear in x?
(defun how-many (e x)
  (cond ((endp x) 0)
        ((equal e (car x)) (+ 1 (how-many e (cdr x))))
        (t (how-many e (cdr x)))))


The admission of HOW-MANY is trivial, using the relation O< (which
is known to be well-founded on the domain recognized by O-P) and the
measure (ACL2-COUNT X).  We observe that the type of HOW-MANY is described
by the theorem (AND (INTEGERP (HOW-MANY E X)) (<= 0 (HOW-MANY E X))).
We used primitive type reasoning.

Summary
Form:  ( DEFUN HOW-MANY ...)
Rules: ((:FAKE-RUNE-FOR-TYPE-SET NIL))


how-many

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)


In [5]:
(how-many 'a '(a b a c a))

3

**Key insight:** Two lists are permutations of each other if and only if every element appears the same number of times in both. The `how-many` approach (from `books/sorting/convert-perm-to-how-many.lisp`) is often easier to work with than the recursive `perm` predicate.

## 2. Insertion Sort

*Source: `books/sorting/isort.lisp`*

Insertion sort works by repeatedly inserting each element into its correct position in an already-sorted list.

In [6]:
; Insert element e into sorted list x
(defun insert-sort (e x)
  (if (endp x)
      (cons e x)
    (if (lexorder e (car x))
        (cons e x)
      (cons (car x)
            (insert-sort e (cdr x))))))


The admission of INSERT-SORT is trivial, using the relation O< (which
is known to be well-founded on the domain recognized by O-P) and the
measure (ACL2-COUNT X).  We observe that the type of INSERT-SORT is
described by the theorem (CONSP (INSERT-SORT E X)).  We used primitive
type reasoning.

Summary
Form:  ( DEFUN INSERT-SORT ...)
Rules: ((:FAKE-RUNE-FOR-TYPE-SET NIL))


insert-sort

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)


In [7]:
; Sort by repeated insertion
(defun isort (x)
  (if (endp x)
      nil
    (insert-sort (car x)
                 (isort (cdr x)))))


The admission of ISORT is trivial, using the relation O< (which is
known to be well-founded on the domain recognized by O-P) and the measure
(ACL2-COUNT X).  We observe that the type of ISORT is described by
the theorem (OR (CONSP (ISORT X)) (EQUAL (ISORT X) NIL)).  We used
the :type-prescription rule INSERT-SORT.

Summary
Form:  ( DEFUN ISORT ...)
Rules: ((:TYPE-PRESCRIPTION INSERT-SORT))


isort

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)


In [8]:
; Let's test it
(isort '(3 1 4 1 5 9 2 6))

(1 1 2 3 4 5 6 9)

### Proving Insertion Sort Correct

We need three theorems:

In [9]:
; 1. The output is ordered
(defthm orderedp-insert-sort
  (implies (orderedp x)
           (orderedp (insert-sort e x))))


Name the formula above *1.

Perhaps we can prove *1 by induction.  Two induction schemes are suggested
by this conjecture.  These merge into one derived induction scheme.

We will induct according to a scheme suggested by (INSERT-SORT E X),
while accommodating (ORDEREDP X).

These suggestions were produced using the :induction rules INSERT-SORT
and ORDEREDP.  If we let (:P E X) denote *1 above then the induction
scheme we'll use is
(AND (IMPLIES (AND (NOT (ENDP X))
                   (NOT (LEXORDER E (CAR X)))
                   (:P E (CDR X)))
              (:P E X))
     (IMPLIES (AND (NOT (ENDP X))
                   (LEXORDER E (CAR X)))
              (:P E X))
     (IMPLIES (ENDP X) (:P E X))).
This induction is justified by the same argument used to admit INSERT-SORT.
When applied to the goal at hand the above induction scheme produces
four nontautological subgoals.

Subgoal *1/4
(IMPLIES (AND (NOT (ENDP X))
              (NOT (LEXORDER E (CAR X)))
              (ORDEREDP (INSER

orderedp-insert-sort

Time:  0.01 seconds (prove: 0.00, print: 0.00, other: 0.00)
Prover steps counted:  2051


In [10]:
(defthm orderedp-isort
  (orderedp (isort x)))


Name the formula above *1.

Perhaps we can prove *1 by induction.  One induction scheme is suggested
by this conjecture.  

We will induct according to a scheme suggested by (ISORT X).

This suggestion was produced using the :induction rule ISORT.  If we
let (:P X) denote *1 above then the induction scheme we'll use is
(AND (IMPLIES (AND (NOT (ENDP X)) (:P (CDR X)))
              (:P X))
     (IMPLIES (ENDP X) (:P X))).
This induction is justified by the same argument used to admit ISORT.
When applied to the goal at hand the above induction scheme produces
two nontautological subgoals.

Subgoal *1/2
(IMPLIES (AND (NOT (ENDP X))
              (ORDEREDP (ISORT (CDR X))))
         (ORDEREDP (ISORT X))).

By the simple :definition ENDP we reduce the conjecture to

Subgoal *1/2'
(IMPLIES (AND (CONSP X)
              (ORDEREDP (ISORT (CDR X))))
         (ORDEREDP (ISORT X))).

But simplification reduces this to T, using the :definition ISORT,
the :rewrite rule ORDEREDP-INSERT-SORT and the :

orderedp-isort

Time:  0.01 seconds (prove: 0.00, print: 0.00, other: 0.00)
Prover steps counted:  257


In [11]:
; 2. The output is a proper list
(defthm true-listp-isort
  (true-listp (isort x)))


Name the formula above *1.

Perhaps we can prove *1 by induction.  One induction scheme is suggested
by this conjecture.  

We will induct according to a scheme suggested by (ISORT X).

This suggestion was produced using the :induction rule ISORT.  If we
let (:P X) denote *1 above then the induction scheme we'll use is
(AND (IMPLIES (AND (NOT (ENDP X)) (:P (CDR X)))
              (:P X))
     (IMPLIES (ENDP X) (:P X))).
This induction is justified by the same argument used to admit ISORT.
When applied to the goal at hand the above induction scheme produces
two nontautological subgoals.

Subgoal *1/2
(IMPLIES (AND (NOT (ENDP X))
              (TRUE-LISTP (ISORT (CDR X))))
         (TRUE-LISTP (ISORT X))).

By the simple :definition ENDP we reduce the conjecture to

Subgoal *1/2'
(IMPLIES (AND (CONSP X)
              (TRUE-LISTP (ISORT (CDR X))))
         (TRUE-LISTP (ISORT X))).

This simplifies, using the :definition ISORT, to

Subgoal *1/2''
(IMPLIES (AND (CONSP X)
              (TRU

true-listp-isort

Time:  0.01 seconds (prove: 0.01, print: 0.01, other: 0.00)
Prover steps counted:  1189


In [12]:
; 3. Element counts are preserved
(defthm how-many-insert-sort
  (equal (how-many a (insert-sort e x))
         (if (equal a e)
             (+ 1 (how-many a x))
           (how-many a x))))


This simplifies, using trivial observations, to the following two conjectures.

Subgoal 2
(IMPLIES (EQUAL A E)
         (EQUAL (HOW-MANY A (INSERT-SORT E X))
                (+ 1 (HOW-MANY A X)))).

This simplifies, using trivial observations, to

Subgoal 2'
(EQUAL (HOW-MANY A (INSERT-SORT A X))
       (+ 1 (HOW-MANY A X))).

Name the formula above *1.

Subgoal 1
(IMPLIES (NOT (EQUAL A E))
         (EQUAL (HOW-MANY A (INSERT-SORT E X))
                (HOW-MANY A X))).

Normally we would attempt to prove Subgoal 1 by induction.  However,
we prefer in this instance to focus on the original input conjecture
rather than this simplified special case.  We therefore abandon our
previous work on this conjecture and reassign the name *1 to the original
conjecture.  (See :DOC otf-flg.)

Perhaps we can prove *1 by induction.  Three induction schemes are
suggested by this conjecture.  Subsumption reduces that number to two.
These merge into one derived induction scheme.  

We will induct accordi

how-many-insert-sort

Time:  0.01 seconds (prove: 0.01, print: 0.00, other: 0.00)
Prover steps counted:  2988


In [13]:
(defthm how-many-isort
  (equal (how-many e (isort x))
         (how-many e x)))


Name the formula above *1.

Perhaps we can prove *1 by induction.  Two induction schemes are suggested
by this conjecture.  Subsumption reduces that number to one.  

We will induct according to a scheme suggested by (HOW-MANY E X), while
accommodating (ISORT X).

These suggestions were produced using the :induction rules HOW-MANY
and ISORT.  If we let (:P E X) denote *1 above then the induction scheme
we'll use is
(AND (IMPLIES (AND (NOT (ENDP X))
                   (NOT (EQUAL E (CAR X)))
                   (:P E (CDR X)))
              (:P E X))
     (IMPLIES (AND (NOT (ENDP X))
                   (EQUAL E (CAR X))
                   (:P E (CDR X)))
              (:P E X))
     (IMPLIES (ENDP X) (:P E X))).
This induction is justified by the same argument used to admit HOW-MANY.
When applied to the goal at hand the above induction scheme produces
three nontautological subgoals.

Subgoal *1/3
(IMPLIES (AND (NOT (ENDP X))
              (NOT (EQUAL E (CAR X)))
              (EQUAL (HO

how-many-isort

Time:  0.01 seconds (prove: 0.00, print: 0.00, other: 0.00)
Prover steps counted:  698


With `orderedp-isort` and `how-many-isort`, we have a **complete correctness proof** for insertion sort. The output is sorted and contains exactly the same elements as the input.

## 3. Merge Sort

*Source: `books/sorting/msort.lisp`*

Merge sort splits the list into halves, sorts each recursively, and merges the results. The split uses `evens` and `odds` to avoid computing length.

### Termination Challenge
Unlike insertion sort (which recurs on `(cdr x)`), merge sort recurs on `(evens x)` and `(odds x)`. We need to prove these are **strictly smaller** than `x`.

In [15]:
; Split into even-indexed and odd-indexed elements
(defun my-evens (x)
  (cond ((endp x) nil)
        ((endp (cdr x)) (list (car x)))
        (t (cons (car x) (my-evens (cddr x))))))


For the admission of MY-EVENS we will use the relation O< (which is
known to be well-founded on the domain recognized by O-P) and the measure
(ACL2-COUNT X).  The non-trivial part of the measure conjecture is

Goal
(IMPLIES (AND (NOT (ENDP X))
              (NOT (ENDP (CDR X))))
         (O< (ACL2-COUNT (CDDR X))
             (ACL2-COUNT X))).

By the simple :definition ENDP we reduce the conjecture to

Goal'
(IMPLIES (AND (CONSP X) (CONSP (CDR X)))
         (O< (ACL2-COUNT (CDDR X))
             (ACL2-COUNT X))).

This simplifies, using the :definitions O-FINP and O< and the :type-
prescription rule ACL2-COUNT, to

Goal''
(IMPLIES (AND (CONSP X) (CONSP (CDR X)))
         (< (ACL2-COUNT (CDDR X))
            (ACL2-COUNT X))).

But simplification reduces this to T, using linear arithmetic, primitive
type reasoning, the :linear rule ACL2-COUNT-CAR-CDR-LINEAR and the
:type-prescription rule ACL2-COUNT.

Q.E.D.

That completes the proof of the measure theorem for MY-EVENS.  Thus,
we admit

my-evens

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)
Prover steps counted:  528


In [16]:
; odds = evens of cdr
(defun my-odds (x)
  (my-evens (cdr x)))


Since MY-ODDS is non-recursive, its admission is trivial.  We observe
that the type of MY-ODDS is described by the theorem 
(TRUE-LISTP (MY-ODDS X)).  We used the :type-prescription rule MY-EVENS.

Summary
Form:  ( DEFUN MY-ODDS ...)
Rules: ((:TYPE-PRESCRIPTION MY-EVENS))


my-odds

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)


In [17]:
; Merge two sorted lists
(defun merge2 (x y)
  (declare (xargs :measure (+ (acl2-count x) (acl2-count y))))
  (if (endp x)
      y
    (if (endp y)
        x
      (if (lexorder (car x) (car y))
          (cons (car x) (merge2 (cdr x) y))
        (cons (car y) (merge2 x (cdr y)))))))


For the admission of MERGE2 we will use the relation O< (which is known
to be well-founded on the domain recognized by O-P) and the measure
(+ (ACL2-COUNT X) (ACL2-COUNT Y)).  The non-trivial part of the measure
conjecture is

Goal
(AND (O-P (+ (ACL2-COUNT X) (ACL2-COUNT Y)))
     (IMPLIES (AND (NOT (ENDP X))
                   (NOT (ENDP Y))
                   (NOT (LEXORDER (CAR X) (CAR Y))))
              (O< (+ (ACL2-COUNT X) (ACL2-COUNT (CDR Y)))
                  (+ (ACL2-COUNT X) (ACL2-COUNT Y))))
     (IMPLIES (AND (NOT (ENDP X))
                   (NOT (ENDP Y))
                   (LEXORDER (CAR X) (CAR Y)))
              (O< (+ (ACL2-COUNT (CDR X)) (ACL2-COUNT Y))
                  (+ (ACL2-COUNT X) (ACL2-COUNT Y))))).

By the simple :definition ENDP we reduce the conjecture to the following
three conjectures.

Subgoal 3
(O-P (+ (ACL2-COUNT X) (ACL2-COUNT Y))).

But simplification reduces this to T, using the :compound-recognizer
rule NATP-COMPOUND-RECOGNIZER, the :definitio

merge2

Time:  0.01 seconds (prove: 0.00, print: 0.00, other: 0.00)
Prover steps counted:  1172


Notice the `:measure` declaration — `merge2` takes two arguments that both shrink, so we need to tell ACL2 that the *sum* of their sizes decreases.

In [18]:
; Termination lemma: evens of a 2+ element list is smaller
(defthm acl2-count-evens-strong
  (implies (and (consp x) (consp (cdr x)))
           (< (acl2-count (my-evens x)) (acl2-count x)))
  :rule-classes :linear)


The destructor terms (CAR X) and (CDR X) can be eliminated by using
CAR-CDR-ELIM to replace X by (CONS X1 X2), (CAR X) by X1 and (CDR X)
by X2.  This produces the following goal.

Goal'
(IMPLIES (AND (CONSP (CONS X1 X2)) (CONSP X2))
         (< (ACL2-COUNT (MY-EVENS (CONS X1 X2)))
            (ACL2-COUNT (CONS X1 X2)))).

This simplifies, using the :definition ACL2-COUNT, primitive type reasoning
and the :rewrite rules CAR-CONS and CDR-CONS, to

Goal''
(IMPLIES (CONSP X2)
         (< (ACL2-COUNT (MY-EVENS (CONS X1 X2)))
            (+ 1 (ACL2-COUNT X1) (ACL2-COUNT X2)))).

Normally we would attempt to prove Goal'' by induction.  However, we
prefer in this instance to focus on the original input conjecture rather
than this simplified special case.  We therefore abandon our previous
work on this conjecture and reassign the name *1 to the original conjecture.
(See :DOC otf-flg.)

Perhaps we can prove *1 by induction.  Two induction schemes are suggested
by this conjecture.  We will choos

acl2-count-evens-strong

Time:  0.02 seconds (prove: 0.02, print: 0.01, other: 0.00)
Prover steps counted:  12266


In [19]:
; The merge sort function
(defun msort (x)
  (if (endp x)
      nil
    (if (endp (cdr x))
        (list (car x))
      (merge2 (msort (my-evens x))
              (msort (my-odds x))))))


For the admission of MSORT we will use the relation O< (which is known
to be well-founded on the domain recognized by O-P) and the measure
(ACL2-COUNT X).  The non-trivial part of the measure conjecture is

Goal
(AND (IMPLIES (AND (NOT (ENDP X))
                   (NOT (ENDP (CDR X))))
              (O< (ACL2-COUNT (MY-ODDS X))
                  (ACL2-COUNT X)))
     (IMPLIES (AND (NOT (ENDP X))
                   (NOT (ENDP (CDR X))))
              (O< (ACL2-COUNT (MY-EVENS X))
                  (ACL2-COUNT X)))).

By the simple :definitions ENDP and MY-ODDS we reduce the conjecture
to the following two conjectures.

Subgoal 2
(IMPLIES (AND (CONSP X) (CONSP (CDR X)))
         (O< (ACL2-COUNT (MY-EVENS (CDR X)))
             (ACL2-COUNT X))).

This simplifies, using the :definitions O-FINP and O< and the :type-
prescription rule ACL2-COUNT, to

Subgoal 2'
(IMPLIES (AND (CONSP X) (CONSP (CDR X)))
         (< (ACL2-COUNT (MY-EVENS (CDR X)))
            (ACL2-COUNT X))).

The destructor 

msort

Time:  0.09 seconds (prove: 0.06, print: 0.03, other: 0.00)
Prover steps counted:  34580


In [20]:
(msort '(3 1 4 1 5 9 2 6))

(1 1 2 3 4 5 6 9)

### Correctness of Merge Sort

In [21]:
; The output is ordered
(defthm orderedp-merge2
  (implies (and (orderedp x) (orderedp y))
           (orderedp (merge2 x y))))


Name the formula above *1.

Perhaps we can prove *1 by induction.  Three induction schemes are
suggested by this conjecture.  These merge into one derived induction
scheme.  

We will induct according to a scheme suggested by (MERGE2 X Y), while
accommodating (ORDEREDP Y) and (ORDEREDP X).

These suggestions were produced using the :induction rules MERGE2 and
ORDEREDP.  If we let (:P X Y) denote *1 above then the induction scheme
we'll use is
(AND (IMPLIES (AND (NOT (ENDP X))
                   (NOT (ENDP Y))
                   (NOT (LEXORDER (CAR X) (CAR Y)))
                   (:P X (CDR Y)))
              (:P X Y))
     (IMPLIES (AND (NOT (ENDP X))
                   (NOT (ENDP Y))
                   (LEXORDER (CAR X) (CAR Y))
                   (:P (CDR X) Y))
              (:P X Y))
     (IMPLIES (AND (NOT (ENDP X)) (ENDP Y))
              (:P X Y))
     (IMPLIES (ENDP X) (:P X Y))).
This induction is justified by the same argument used to admit MERGE2.
When applied to the goal a

orderedp-merge2

Time:  0.03 seconds (prove: 0.02, print: 0.01, other: 0.00)
Prover steps counted:  10380


In [22]:
(defthm orderedp-msort
  (orderedp (msort x)))


Name the formula above *1.

Perhaps we can prove *1 by induction.  One induction scheme is suggested
by this conjecture.  

We will induct according to a scheme suggested by (MSORT X).

This suggestion was produced using the :induction rule MSORT.  If we
let (:P X) denote *1 above then the induction scheme we'll use is
(AND (IMPLIES (AND (NOT (ENDP X))
                   (NOT (ENDP (CDR X)))
                   (:P (MY-EVENS X))
                   (:P (MY-ODDS X)))
              (:P X))
     (IMPLIES (AND (NOT (ENDP X)) (ENDP (CDR X)))
              (:P X))
     (IMPLIES (ENDP X) (:P X))).
This induction is justified by the same argument used to admit MSORT.
When applied to the goal at hand the above induction scheme produces
three nontautological subgoals.

Subgoal *1/3
(IMPLIES (AND (NOT (ENDP X))
              (NOT (ENDP (CDR X)))
              (ORDEREDP (MSORT (MY-EVENS X)))
              (ORDEREDP (MSORT (MY-ODDS X))))
         (ORDEREDP (MSORT X))).

By the simple :definitions EN

orderedp-msort

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)
Prover steps counted:  772


In [23]:
; Element counts are preserved through merge
(defthm how-many-merge2
  (equal (how-many e (merge2 x y))
         (+ (how-many e x) (how-many e y))))


Name the formula above *1.

Perhaps we can prove *1 by induction.  Three induction schemes are
suggested by this conjecture.  These merge into one derived induction
scheme.  

We will induct according to a scheme suggested by (MERGE2 X Y), while
accommodating (HOW-MANY E Y) and (HOW-MANY E X).

These suggestions were produced using the :induction rules HOW-MANY
and MERGE2.  If we let (:P E X Y) denote *1 above then the induction
scheme we'll use is
(AND (IMPLIES (AND (NOT (ENDP X))
                   (NOT (ENDP Y))
                   (NOT (LEXORDER (CAR X) (CAR Y)))
                   (:P E X (CDR Y)))
              (:P E X Y))
     (IMPLIES (AND (NOT (ENDP X))
                   (NOT (ENDP Y))
                   (LEXORDER (CAR X) (CAR Y))
                   (:P E (CDR X) Y))
              (:P E X Y))
     (IMPLIES (AND (NOT (ENDP X)) (ENDP Y))
              (:P E X Y))
     (IMPLIES (ENDP X) (:P E X Y))).
This induction is justified by the same argument used to admit MERGE2.
When app

how-many-merge2

Time:  0.01 seconds (prove: 0.01, print: 0.00, other: 0.00)
Prover steps counted:  3145


In [24]:
; Key lemma: evens + odds reconstitute the original
(defthm how-many-evens-and-odds
  (implies (consp x)
           (equal (+ (how-many e (my-evens x))
                     (how-many e (my-odds x)))
                  (how-many e x))))


ACL2 Warning [Non-rec] in ( DEFTHM HOW-MANY-EVENS-AND-ODDS ...):  A
:REWRITE rule generated from HOW-MANY-EVENS-AND-ODDS will be triggered
only by terms containing the function symbol MY-ODDS, which has a non-
recursive definition.  Unless this definition is disabled, this rule
is unlikely ever to be used.


ACL2 Warning [Subsume] in ( DEFTHM HOW-MANY-EVENS-AND-ODDS ...):  The
previously added rule COMMUTATIVITY-OF-+ subsumes a newly proposed
:REWRITE rule generated from HOW-MANY-EVENS-AND-ODDS, in the sense
that the old rule rewrites a more general target.  Because the new
rule will be tried first, it may nonetheless find application.


By the simple :definition MY-ODDS we reduce the conjecture to

Goal'
(IMPLIES (CONSP X)
         (EQUAL (+ (HOW-MANY E (MY-EVENS X))
                   (HOW-MANY E (MY-EVENS (CDR X))))
                (HOW-MANY E X))).

This simplifies, using the :definition HOW-MANY (if-intro), to the
following two conjectures.

Subgoal 2
(IMPLIES (AND (CONSP X) (EQU

how-many-evens-and-odds

Time:  0.02 seconds (prove: 0.02, print: 0.01, other: 0.00)
Prover steps counted:  8753


In [25]:
(defthm how-many-msort
  (equal (how-many e (msort x))
         (how-many e x)))


Name the formula above *1.

Perhaps we can prove *1 by induction.  Two induction schemes are suggested
by this conjecture.  By considering those suggested by the largest
number of non-primitive recursive functions, we narrow the field to
one.  

We will induct according to a scheme suggested by (MSORT X).

This suggestion was produced using the :induction rule MSORT.  If we
let (:P E X) denote *1 above then the induction scheme we'll use is
(AND (IMPLIES (AND (NOT (ENDP X))
                   (NOT (ENDP (CDR X)))
                   (:P E (MY-EVENS X))
                   (:P E (MY-ODDS X)))
              (:P E X))
     (IMPLIES (AND (NOT (ENDP X)) (ENDP (CDR X)))
              (:P E X))
     (IMPLIES (ENDP X) (:P E X))).
This induction is justified by the same argument used to admit MSORT.
When applied to the goal at hand the above induction scheme produces
three nontautological subgoals.

Subgoal *1/3
(IMPLIES (AND (NOT (ENDP X))
              (NOT (ENDP (CDR X)))
              (EQUAL

## 4. Bubble Sort

*Source: `books/sorting/bsort.lisp`*

Bubble sort repeatedly passes through the list, swapping adjacent elements that are out of order. The challenge is proving **termination** — we need a measure that strictly decreases with each pass.

The book uses `bnext-size`: the total number of inversions (pairs where a smaller element follows a larger one).

In [ ]:
; One pass of bubble sort
(defun bnext (x)
  (declare (xargs :measure (len x)))
  (cond ((endp x) x)
        ((endp (cdr x)) x)
        ((lexorder (car x) (cadr x))
         (cons (car x) (bnext (cdr x))))
        (t (cons (cadr x)
                 (bnext (cons (car x) (cddr x)))))))

In [ ]:
; Count inversions as a termination measure
(defun how-many-smaller (e x)
  (cond ((endp x) 0)
        ((equal e (car x)) (how-many-smaller e (cdr x)))
        ((lexorder (car x) e) (+ 1 (how-many-smaller e (cdr x))))
        (t (how-many-smaller e (cdr x)))))

In [ ]:
(defun bnext-size (x)
  (cond ((endp x) 0)
        (t (+ (how-many-smaller (car x) (cdr x))
              (bnext-size (cdr x))))))

In [ ]:
; Bubble sort: iterate bnext until ordered
(defun bsort (x)
  (declare (xargs :measure (bnext-size x)))
  (if (orderedp x)
      x
    (bsort (bnext x))))

In [ ]:
(bsort '(3 1 4 1 5 9 2 6))

## 5. Quicksort

*Source: `books/sorting/qsort.lisp`*

Quicksort partitions the list around a pivot, sorts each partition, and concatenates. The `filter` function selects elements based on a comparison relation.

In [ ]:
; Ordering relation selector
(defun rel (fn i j)
  (case fn
    (LT  (and (lexorder i j) (not (equal i j))))
    (LTE (lexorder i j))
    (GT  (and (lexorder j i) (not (equal i j))))
    (otherwise (lexorder j i))))

In [ ]:
; Filter elements by relation to pivot
(defun qfilter (fn x e)
  (cond ((endp x) nil)
        ((rel fn (car x) e)
         (cons (car x) (qfilter fn (cdr x) e)))
        (t (qfilter fn (cdr x) e))))

In [ ]:
; Quicksort
(defun qsort (x)
  (cond ((endp x) nil)
        ((endp (cdr x)) (list (car x)))
        (t (append (qsort (qfilter 'LT (cdr x) (car x)))
                   (cons (car x)
                         (qsort (qfilter 'GTE (cdr x) (car x))))))))

In [ ]:
(qsort '(3 1 4 1 5 9 2 6))

## 6. All Sorts Are Equivalent

*Source: `books/sorting/equisort.lisp`*

Since each sorting algorithm produces an ordered list that is a permutation of the input, they must all produce the **same output** (assuming a total order). This is because:

> **Theorem:** If two lists are both ordered and have the same element counts, they are equal.

This means `isort`, `msort`, `bsort`, and `qsort` all compute the same function!

In [ ]:
; Two sorted lists with the same element counts are equal
(defthm orderedp-same-how-many-equal
  (implies (and (orderedp x)
                (orderedp y)
                (true-listp x)
                (true-listp y))
           (iff (equal x y)
                (equal (how-many-list x)
                       (how-many-list y)))))

The `equisort.lisp` book proves:
```
(defthm isort-is-msort
  (equal (isort x) (msort x)))

(defthm isort-is-qsort
  (equal (isort x) (qsort x)))
```

This is a profound result: the *specification* (ordered + permutation) uniquely determines the function.

## 7. Complexity Notes

| Algorithm | Time Complexity | Space | Key Insight |
|---|---|---|---|
| Insertion sort | $O(n^2)$ worst, $O(n)$ best | $O(1)$ | Simple, good for small/nearly-sorted |
| Merge sort | $O(n \log n)$ always | $O(n)$ | Divide-and-conquer, stable |
| Bubble sort | $O(n^2)$ worst and average | $O(1)$ | Simple but slow |
| Quicksort | $O(n \log n)$ average, $O(n^2)$ worst | $O(\log n)$ | Fast in practice |

## 8. Exercises

**Exercise 1:** Define a predicate `sortedp-rev` that checks if a list is sorted in reverse (descending) order. Prove that `(sortedp-rev (reverse (isort x)))` holds.

**Exercise 2:** The `merge2` function uses a two-argument measure `(+ (acl2-count x) (acl2-count y))`. Explain why `(acl2-count x)` alone would not suffice.

**Exercise 3:** Define `count-inversions` that counts the number of pairs `(i,j)` with `i < j` but `(nth i x) > (nth j x)`. Prove that `(count-inversions (isort x)) = 0`.

**Exercise 4 (Challenge):** Define a stable merge sort — one that preserves the relative order of equal elements. Prove stability as a formal theorem.

---
**Next:** [Module 4 — Data Structures](04_data_structures.ipynb) — Binary search trees and balanced trees, verified.